# Algoritmos de Busca e Tabelas Hash
## Implementação em C e análise experimental com Python

Neste notebook, os **conceitos principais serão implementados em linguagem C**. Python será utilizado apenas quando for útil para:

- ler arquivos CSV produzidos pelos programas C;
- organizar os resultados experimentais;
- calcular médias;
- visualizar tendências por meio de gráficos.

A proposta é que você desenvolva duas competências em paralelo:

1. **implementar e analisar algoritmos em C**;
2. começar a usar **Python como ferramenta de análise de dados**.

---

## Objetivos de aprendizagem

Ao final das atividades, você deverá ser capaz de:

- implementar **busca sequencial** em C;
- implementar **busca binária recursiva** em C;
- explicar por que a busca binária exige dados ordenados;
- comparar experimentalmente busca sequencial e busca binária;
- analisar quando compensa **ordenar uma coleção para depois realizar buscas binárias**;
- implementar funções hash pelos métodos da **divisão** e **folding**;
- identificar e contabilizar **colisões**;
- tratar colisões por:
  - **chaining** com listas ligadas em C;
  - **endereçamento aberto com sondagem quadrática**;
- compreender o conceito de **fator de carga**;
- comparar a distribuição e o custo de diferentes estratégias de hashing;
- usar `pandas` e `matplotlib` para interpretar os resultados produzidos pelos programas C.

> **Regra da atividade:** busca, hashing, tratamento de colisões e estruturas de dados devem ser implementados em **C**. Python não deve substituir essas implementações.

# 0. Executando programas C no Google Colab

O Colab usa Python por padrão, mas podemos criar, compilar e executar programas C.

Usaremos três recursos:

### `%%writefile`

A primeira linha

```text
%%writefile programa.c
```

faz com que o conteúdo da célula seja gravado no arquivo `programa.c`.

### `gcc`

Podemos compilar C executando um comando do sistema operacional:

```text
!gcc -Wall -Wextra -O2 -std=c11 programa.c -o programa
```

Algumas opções:

- `-Wall -Wextra`: mostram avisos úteis do compilador;
- `-O2`: habilita otimizações;
- `-std=c11`: utiliza o padrão C11;
- `-o programa`: define o nome do executável.

### execução

```text
!./programa
```

O caractere `!` permite executar comandos do sistema operacional a partir do notebook.

# Parte I — Algoritmos de busca

Antes de comparar tempos, vamos registrar uma métrica mais diretamente ligada ao algoritmo: o número de **elementos inspecionados** durante uma busca.

Usaremos:

```c
typedef struct {
    int indice;
    long long inspecoes;
} ResultadoBusca;
```

- `indice >= 0`: posição em que a chave foi encontrada;
- `indice == -1`: chave não encontrada;
- `inspecoes`: quantidade de posições do vetor examinadas.

Essa métrica é mais estável do que tempo de execução, pois não depende diretamente da velocidade do computador.

In [ ]:
%%writefile busca.h
#ifndef BUSCA_H
#define BUSCA_H

typedef struct {
    int indice;
    long long inspecoes;
} ResultadoBusca;

ResultadoBusca busca_sequencial(const int v[], int n, int chave);

ResultadoBusca busca_binaria_recursiva(
    const int v[],
    int inicio,
    int fim,
    int chave
);

#endif

# Exercício Prático 1 — Busca sequencial

Implemente em C o algoritmo de busca sequencial.

Pseudocódigo:

```text
busca_sequencial(L, chave)
    para i = 0 até tamanho(L)-1
        inspecionar L[i]

        se L[i] == chave
            retornar i

    retornar -1
```

## Melhoria em relação à versão booleana

Em vez de retornar apenas verdadeiro/falso, sua função deve retornar também o **índice** em que a chave foi encontrada.

Isso permite distinguir:

- chave encontrada na primeira posição;
- chave encontrada perto do final;
- chave ausente.

Além disso, conte quantos elementos foram inspecionados.

### Casos que você deve prever

- vetor vazio;
- chave na primeira posição;
- chave na última posição;
- chave inexistente;
- valores repetidos.

Se houver valores repetidos, a busca sequencial deve retornar a **primeira ocorrência**.

In [ ]:
%%writefile busca_sequencial.c
#include "busca.h"

ResultadoBusca busca_sequencial(const int v[], int n, int chave) {
    ResultadoBusca r;
    r.indice = -1;
    r.inspecoes = 0;

    /* TODO:
       Percorra o vetor.
       Para cada posição examinada, incremente r.inspecoes.
       Ao encontrar a chave, armazene seu índice e retorne.
    */

    return r;
}

## Testando antes de medir desempenho

Antes de realizar qualquer benchmark, devemos verificar a **correção** do algoritmo.

O programa abaixo usa testes determinísticos. Não altere os valores esperados para "fazer o teste passar": corrija a função.

In [ ]:
%%writefile teste_sequencial.c
#include <stdio.h>
#include "busca.h"

void testa(const int v[], int n, int chave, int esperado) {
    ResultadoBusca r = busca_sequencial(v, n, chave);

    printf("chave=%d | indice=%d | esperado=%d | inspecoes=%lld | %s\n",
           chave,
           r.indice,
           esperado,
           r.inspecoes,
           r.indice == esperado ? "OK" : "ERRO");
}

int main(void) {
    int v[] = {15, 22, 37, 41, 58, 63, 79, 84};
    int n = sizeof(v) / sizeof(v[0]);

    testa(v, n, 15, 0);   // primeira posição
    testa(v, n, 41, 3);   // posição intermediária
    testa(v, n, 84, 7);   // última posição
    testa(v, n, 99, -1);  // ausente

    int repetidos[] = {4, 7, 7, 7, 20};
    testa(repetidos, 5, 7, 1);

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O3 -std=c11 \
    busca_sequencial.c teste_sequencial.c \
    -o teste_sequencial

!./teste_sequencial

Chave: 15
Vetor: 
15 22 37 41 58 63 79 84 
chave=15 | indice=0 | esperado=0 | inspecoes=1 | OK
Chave: 41
Vetor: 
15 22 37 41 58 63 79 84 
chave=41 | indice=3 | esperado=3 | inspecoes=4 | OK
Chave: 84
Vetor: 
15 22 37 41 58 63 79 84 
chave=84 | indice=7 | esperado=7 | inspecoes=8 | OK
Chave: 99
Vetor: 
15 22 37 41 58 63 79 84 
chave=99 | indice=-1 | esperado=-1 | inspecoes=8 | OK
Chave: 7
Vetor: 
4 7 7 7 20 
chave=7 | indice=1 | esperado=1 | inspecoes=2 | OK


## Questões — Busca sequencial

1. Quantos elementos são inspecionados quando a chave está na primeira posição?
2. E quando está na última posição?
3. Quantos elementos são inspecionados quando a chave não existe?
4. Qual é o melhor caso?
5. Qual é o pior caso?
6. Qual complexidade assintótica representa o pior caso?

# Exercício Prático 2 — Busca binária recursiva

A busca binária reduz o intervalo de busca aproximadamente pela metade a cada chamada.

Pseudocódigo:

```text
buscaBinaria(L, inicio, fim, chave)

    se inicio > fim
        retornar -1

    meio = inicio + (fim - inicio) / 2
    inspecionar L[meio]

    se L[meio] == chave
        retornar meio

    se L[meio] > chave
        retornar buscaBinaria(L, inicio, meio-1, chave)

    retornar buscaBinaria(L, meio+1, fim, chave)
```

## Pré-condição fundamental

> A busca binária só funciona corretamente se o vetor estiver **ordenado** segundo o mesmo critério usado nas comparações.

Essa pré-condição não é um detalhe de implementação: faz parte do algoritmo.

### Por que usamos

```c
inicio + (fim - inicio) / 2
```

em vez de

```c
(inicio + fim) / 2
```

?

A primeira expressão evita um possível overflow de `inicio + fim` para índices muito grandes.

In [ ]:
%%writefile busca_binaria.c
#include "busca.h"

ResultadoBusca busca_binaria_recursiva(
    const int v[],
    int inicio,
    int fim,
    int chave
) {
    ResultadoBusca r;
    r.indice = -1;
    r.inspecoes = 0;

    /* TODO:
       Implemente a busca binária recursiva.

       ATENÇÃO:
       Como cada chamada recursiva cria um novo ResultadoBusca,
       você deve somar as inspeções realizadas na chamada atual
       às inspeções retornadas pela chamada seguinte.
    */

    return r;
}

## Teste da busca binária

O vetor abaixo já está ordenado. Depois de validar o algoritmo, experimente alterar a ordem de alguns elementos e observe o que acontece.

Isso ajuda a perceber que a ordenação é realmente uma pré-condição.

In [ ]:
%%writefile teste_binaria.c
#include <stdio.h>
#include "busca.h"

void testa(const int v[], int n, int chave, int esperado) {
    ResultadoBusca r =
        busca_binaria_recursiva(v, 0, n - 1, chave);

    printf("chave=%d | indice=%d | esperado=%d | inspecoes=%lld | %s\n",
           chave,
           r.indice,
           esperado,
           r.inspecoes,
           r.indice == esperado ? "OK" : "ERRO");
}

int main(void) {
    int v[] = {10, 14, 20, 28, 35, 41, 55, 70, 83, 91};
    int n = sizeof(v) / sizeof(v[0]);

    testa(v, n, 10, 0);
    testa(v, n, 35, 4);
    testa(v, n, 91, 9);
    testa(v, n, 50, -1);

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    busca_binaria.c teste_binaria.c \
    -o teste_binaria

!./teste_binaria

## Questões — Busca binária

1. Quantas posições aproximadamente precisam ser examinadas para um vetor de tamanho:
   - 100?
   - 1.000?
   - 1.000.000?
2. Por que o crescimento é muito menor que o da busca sequencial?
3. O que acontece se a lista não estiver ordenada?
4. Qual a relação entre a profundidade da recursão e \(\log_2 n\)?

# Exercício Prático 3 — Busca sequencial × busca binária

Agora compare os dois algoritmos experimentalmente.

Para permitir uma comparação controlada, o benchmark usa vetores ordenados com valores pares:

```text
0, 2, 4, 6, 8, ...
```

As consultas alternam entre:

- **sucesso**: chave existente;
- **fracasso**: chave ímpar, portanto inexistente.

Cada tamanho é testado várias vezes.

O programa C deverá gerar:

```text
resultados_buscas.csv
```

com as colunas:

```text
algoritmo,n,tipo,repeticao,tempo_total_ms,tempo_medio_us,inspecoes_medias
```

> O tempo é medido para um **lote de consultas**, pois uma única busca é rápida demais para ser medida de forma estável.

In [ ]:
%%writefile benchmark_buscas.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include "busca.h"

#define CONSULTAS 1000
#define REPETICOES 5

double tempo_ms(clock_t inicio, clock_t fim) {
    return 1000.0 * (double)(fim - inicio) / CLOCKS_PER_SEC;
}

int main(void) {
    const int tamanhos[] = {100, 500, 1000, 5000, 10000, 20000};
    const int nt = sizeof(tamanhos) / sizeof(tamanhos[0]);

    FILE *arq = fopen("resultados_buscas.csv", "w");
    if (!arq) {
        perror("resultados_buscas.csv");
        return 1;
    }

    fprintf(arq,
        "algoritmo,n,tipo,repeticao,tempo_total_ms,"
        "tempo_medio_us,inspecoes_medias\n");

    srand(42);

    for (int s = 0; s < nt; s++) {
        int n = tamanhos[s];
        int *v = malloc(n * sizeof(int));

        if (!v) {
            fclose(arq);
            return 2;
        }

        for (int i = 0; i < n; i++)
            v[i] = 2 * i;

        for (int tipo = 0; tipo < 2; tipo++) {
            const char *nome_tipo =
                tipo == 0 ? "sucesso" : "fracasso";

            for (int rep = 1; rep <= REPETICOES; rep++) {

                long long total_seq = 0;
                long long total_bin = 0;

                /* Busca sequencial */
                clock_t ini_seq = clock();

                for (int q = 0; q < CONSULTAS; q++) {
                    int pos = rand() % n;
                    int chave =
                        tipo == 0 ? 2 * pos : 2 * pos + 1;

                    ResultadoBusca r =
                        busca_sequencial(v, n, chave);

                    total_seq += r.inspecoes;
                }

                clock_t fim_seq = clock();

                double ms_seq = tempo_ms(ini_seq, fim_seq);

                fprintf(arq,
                    "sequencial,%d,%s,%d,%.6f,%.6f,%.6f\n",
                    n, nome_tipo, rep,
                    ms_seq,
                    1000.0 * ms_seq / CONSULTAS,
                    (double)total_seq / CONSULTAS);

                /* Busca binária */
                clock_t ini_bin = clock();

                for (int q = 0; q < CONSULTAS; q++) {
                    int pos = rand() % n;
                    int chave =
                        tipo == 0 ? 2 * pos : 2 * pos + 1;

                    ResultadoBusca r =
                        busca_binaria_recursiva(
                            v, 0, n - 1, chave);

                    total_bin += r.inspecoes;
                }

                clock_t fim_bin = clock();

                double ms_bin = tempo_ms(ini_bin, fim_bin);

                fprintf(arq,
                    "binaria,%d,%s,%d,%.6f,%.6f,%.6f\n",
                    n, nome_tipo, rep,
                    ms_bin,
                    1000.0 * ms_bin / CONSULTAS,
                    (double)total_bin / CONSULTAS);
            }
        }

        free(v);
    }

    fclose(arq);
    printf("Arquivo resultados_buscas.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    busca_sequencial.c busca_binaria.c benchmark_buscas.c \
    -o benchmark_buscas

!./benchmark_buscas

# Python — lendo resultados com `pandas`

A partir daqui, Python será usado somente para **analisar os dados produzidos pelo programa C**.

A biblioteca `pandas` trabalha com estruturas tabulares chamadas **DataFrames**.

```python
import pandas as pd
df = pd.read_csv("resultados_buscas.csv")
```

`read_csv`:

1. abre o arquivo;
2. identifica as colunas;
3. converte valores numéricos;
4. devolve uma tabela que pode ser filtrada e agrupada.

`head()` exibe as primeiras linhas.

In [ ]:
import pandas as pd

buscas = pd.read_csv("resultados_buscas.csv")

display(buscas.head())
print("Dimensões:", buscas.shape)
print("Algoritmos:", buscas["algoritmo"].unique())
print("Tipos:", buscas["tipo"].unique())

## `groupby` e média das repetições

Como cada experimento foi repetido, devemos resumir os resultados.

```python
df.groupby([...]).agg(...)
```

cria grupos de linhas com características iguais e calcula estatísticas em cada grupo.

No código abaixo, agrupamos por:

- algoritmo;
- tamanho do vetor;
- tipo de busca.

Depois calculamos a média do tempo e do número de posições inspecionadas.

In [ ]:
resumo_buscas = (
    buscas.groupby(["algoritmo", "n", "tipo"])
          .agg(
              tempo_medio_us=("tempo_medio_us", "mean"),
              inspecoes_medias=("inspecoes_medias", "mean"),
          )
          .reset_index()
)

display(resumo_buscas.head(12))

# Python — visualização com `matplotlib`

`matplotlib.pyplot` é uma biblioteca básica de gráficos em Python:

```python
import matplotlib.pyplot as plt
```

Neste gráfico:

- eixo x: tamanho \(n\);
- eixo y: número médio de elementos inspecionados;
- uma curva representa cada algoritmo.

O número de inspeções evidencia muito bem a diferença entre crescimento linear e logarítmico.

In [ ]:
import matplotlib.pyplot as plt

dados = resumo_buscas[resumo_buscas["tipo"] == "fracasso"]

plt.figure(figsize=(8, 5))

for algoritmo in dados["algoritmo"].unique():
    parte = dados[dados["algoritmo"] == algoritmo]

    plt.plot(
        parte["n"],
        parte["inspecoes_medias"],
        marker="o",
        label=algoritmo
    )

plt.xlabel("Tamanho do vetor (n)")
plt.ylabel("Elementos inspecionados em média")
plt.title("Busca malsucedida: sequencial × binária")
plt.legend()
plt.grid()
plt.show()

## Um segundo gráfico: tempo

O gráfico de tempo complementa o gráfico anterior, mas deve ser interpretado com cautela.

O tempo pode variar com:

- processador;
- compilador;
- otimizações;
- cache;
- processos concorrentes;
- resolução do relógio.

Por isso, **não abandone a contagem de inspeções** ao analisar desempenho.

In [ ]:
dados = resumo_buscas[resumo_buscas["tipo"] == "fracasso"]

plt.figure(figsize=(8, 5))

for algoritmo in dados["algoritmo"].unique():
    parte = dados[dados["algoritmo"] == algoritmo]

    plt.plot(
        parte["n"],
        parte["tempo_medio_us"],
        marker="o",
        label=algoritmo
    )

plt.xlabel("Tamanho do vetor (n)")
plt.ylabel("Tempo médio por busca (µs)")
plt.title("Tempo: busca sequencial × busca binária")
plt.legend()
plt.grid()
plt.show()

# Exercício Prático 4 — Vale a pena ordenar antes de usar busca binária?

Uma comparação direta pode ser enganosa:

- busca sequencial funciona em vetor não ordenado;
- busca binária exige vetor ordenado.

Portanto, se os dados ainda não estiverem ordenados, existe um **custo inicial de ordenação**.

Neste exercício, compare:

### Estratégia A

Realizar todas as consultas por busca sequencial em um vetor não ordenado.

### Estratégia B

1. ordenar o vetor uma única vez;
2. realizar todas as consultas por busca binária.

Para isolar a pergunta sobre **custo de busca**, o benchmark pode utilizar `qsort` da biblioteca padrão C apenas para a ordenação. O objetivo aqui **não é substituir os algoritmos de ordenação estudados anteriormente**.

Depois, como desafio, substitua `qsort` por suas próprias implementações de Quick Sort e Merge Sort dos notebooks anteriores.

Teste diferentes quantidades de consultas, por exemplo:

```text
1, 10, 100, 1000, 5000
```

### Pergunta central

> A partir de aproximadamente quantas buscas o custo inicial da ordenação passa a compensar?

In [ ]:
%%writefile benchmark_ordenar_para_buscar.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include "busca.h"

int compara_int(const void *a, const void *b) {
    int x = *(const int *)a;
    int y = *(const int *)b;

    return (x > y) - (x < y);
}

double ms(clock_t inicio, clock_t fim) {
    return 1000.0 * (double)(fim - inicio) / CLOCKS_PER_SEC;
}

int main(void) {
    const int n = 20000;
    const int quantidades[] = {1, 10, 100, 1000, 5000};
    const int nq = sizeof(quantidades) / sizeof(quantidades[0]);

    int *original = malloc(n * sizeof(int));
    int *ordenado = malloc(n * sizeof(int));

    if (!original || !ordenado)
        return 1;

    srand(123);

    for (int i = 0; i < n; i++)
        original[i] = rand();

    FILE *arq = fopen("ordenar_para_buscar.csv", "w");
    if (!arq)
        return 2;

    fprintf(arq, "estrategia,numero_buscas,tempo_total_ms\n");

    for (int k = 0; k < nq; k++) {
        int q = quantidades[k];

        /* Estratégia A: somente busca sequencial */
        clock_t ini_a = clock();

        for (int i = 0; i < q; i++) {
            int chave = original[rand() % n];
            busca_sequencial(original, n, chave);
        }

        clock_t fim_a = clock();

        fprintf(arq, "sequencial,%d,%.6f\n",
                q, ms(ini_a, fim_a));

        /* Estratégia B: ordenar uma vez + q buscas binárias */
        memcpy(ordenado, original, n * sizeof(int));

        clock_t ini_b = clock();

        qsort(ordenado, n, sizeof(int), compara_int);

        for (int i = 0; i < q; i++) {
            int chave = original[rand() % n];

            busca_binaria_recursiva(
                ordenado, 0, n - 1, chave);
        }

        clock_t fim_b = clock();

        fprintf(arq, "ordenar+binaria,%d,%.6f\n",
                q, ms(ini_b, fim_b));
    }

    fclose(arq);
    free(original);
    free(ordenado);

    printf("Arquivo ordenar_para_buscar.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    busca_sequencial.c busca_binaria.c \
    benchmark_ordenar_para_buscar.c \
    -o benchmark_ordenar_para_buscar

!./benchmark_ordenar_para_buscar

## Visualizando o ponto de compensação

Aqui não precisamos de `groupby`, pois cada par estratégia/quantidade aparece uma única vez.

Filtramos cada estratégia e desenhamos uma curva.

In [ ]:
custo = pd.read_csv("ordenar_para_buscar.csv")

display(custo)

plt.figure(figsize=(8, 5))

for estrategia in custo["estrategia"].unique():
    parte = custo[custo["estrategia"] == estrategia]

    plt.plot(
        parte["numero_buscas"],
        parte["tempo_total_ms"],
        marker="o",
        label=estrategia
    )

plt.xlabel("Número de buscas")
plt.ylabel("Tempo total (ms)")
plt.title("Compensa ordenar antes de buscar?")
plt.legend()
plt.grid()
plt.show()

## Desafio de integração com os notebooks de ordenação

Substitua `qsort` por:

1. sua implementação de **Quick Sort**;
2. sua implementação de **Merge Sort**.

Compare três estratégias:

```text
sequencial
quick_sort + busca binária
merge_sort + busca binária
```

Faça isso para diferentes quantidades de consultas.

Essa atividade conecta os dois tópicos do curso: **ordenar** e **buscar**.

# Parte II — Tabelas Hash

Uma tabela hash procura transformar uma chave em um índice da tabela por meio de uma **função de espalhamento**:

\[
h(k) \rightarrow \text{posição}
\]

Uma função muito simples é o método da divisão:

\[
h(k) = k \bmod m
\]

onde:

- \(k\) é a chave;
- \(m\) é o tamanho da tabela.

---

## Colisão

Uma colisão ocorre quando duas chaves diferentes são mapeadas para a mesma posição:

\[
h(k_1)=h(k_2), \quad k_1 \neq k_2.
\]

Colisões são normais em tabelas hash. O problema não é impedir todas as colisões, mas **tratá-las corretamente e manter buscas eficientes**.

---

## Fator de carga

Definimos:

\[
\alpha = \frac{n}{m}
\]

onde:

- \(n\): número de elementos armazenados;
- \(m\): número de posições/buckets.

Quanto maior \(\alpha\), maior tende a ser a pressão por colisões.

Em **endereçamento aberto**, normalmente precisamos de \(\alpha < 1\).

Em **chaining**, é possível ter \(\alpha > 1\), pois cada posição pode armazenar uma lista de elementos.

# Exercício Prático 5 — Método da divisão e visualização de colisões

O exercício original propõe uma tabela de tamanho 10 com muitos valores. Antes de tratar as colisões, é útil observar **onde elas surgem**.

Nesta primeira atividade, implemente apenas a função:

```c
int hash_divisao(int chave, int m);
```

e um pequeno experimento que:

1. gere 20 inteiros aleatórios entre 10 e 1000;
2. calcule `h(chave)`;
3. conte quantas chaves foram direcionadas a cada posição;
4. grave um CSV com:
   - posição;
   - quantidade de chaves mapeadas para ela.

> Neste exercício ainda não estamos construindo uma estrutura que armazena todas as chaves. Estamos estudando a **distribuição da função hash**.

Use `srand(42)` para que os colegas obtenham as mesmas chaves e possam comparar resultados.

In [ ]:
%%writefile hash_divisao_distribuicao.c
#include <stdio.h>
#include <stdlib.h>

#define M 10
#define N_CHAVES 20

int hash_divisao(int chave, int m) {
    /* TODO:
       implemente h(chave) = chave % m
    */
    return 0;
}

int main(void) {
    int contagem[M] = {0};

    srand(42);

    printf("Chaves e posições:\n");

    for (int i = 0; i < N_CHAVES; i++) {
        int chave = 10 + rand() % 991;
        int pos = hash_divisao(chave, M);

        printf("%4d -> %d\n", chave, pos);

        /* TODO:
           incremente a contagem da posição calculada.
        */
    }

    FILE *arq = fopen("distribuicao_divisao.csv", "w");
    if (!arq)
        return 1;

    fprintf(arq, "posicao,quantidade\n");

    for (int i = 0; i < M; i++)
        fprintf(arq, "%d,%d\n", i, contagem[i]);

    fclose(arq);

    printf("Arquivo distribuicao_divisao.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    hash_divisao_distribuicao.c \
    -o hash_divisao_distribuicao

!./hash_divisao_distribuicao

## Python — gráfico de barras da distribuição

Para visualizar quantas chaves chegaram a cada posição, um gráfico de barras é mais adequado do que um gráfico de linhas.

Usamos:

```python
plt.bar(x, y)
```

Cada barra representa um bucket da tabela.

In [ ]:
distrib_div = pd.read_csv("distribuicao_divisao.csv")

display(distrib_div)

plt.figure(figsize=(8, 4))
plt.bar(distrib_div["posicao"], distrib_div["quantidade"])
plt.xlabel("Posição da tabela")
plt.ylabel("Quantidade de chaves")
plt.title("Distribuição — método da divisão")
plt.show()

# Exercício Prático 6 — Hash por divisão com chaining

Agora todas as chaves devem ser armazenadas.

No **chaining**, cada posição da tabela referencia uma lista ligada.

Exemplo:

```text
posição 0 -> 120 -> 340 -> NULL
posição 1 -> NULL
posição 2 -> 42 -> 102 -> 212 -> NULL
...
```

Essa atividade é importante porque reúne:

- arrays;
- ponteiros;
- `struct`;
- alocação dinâmica;
- listas ligadas;
- função hash.

## Estrutura sugerida

```c
typedef struct No {
    int chave;
    struct No *prox;
} No;

typedef struct {
    No **buckets;
    int tamanho;
    int quantidade;
    long long colisoes;
} HashChaining;
```

### Implemente

- criação da tabela;
- inserção;
- busca;
- impressão;
- liberação de memória.

### Critério de colisão

Considere colisão quando uma nova chave é inserida em um bucket que já contém pelo menos um nó.

In [ ]:
%%writefile hash_chaining_divisao.c
#include <stdio.h>
#include <stdlib.h>

typedef struct No {
    int chave;
    struct No *prox;
} No;

typedef struct {
    No **buckets;
    int tamanho;
    int quantidade;
    long long colisoes;
} HashChaining;

int hash_divisao(int chave, int m) {
    return chave % m;
}

HashChaining *cria_hash(int tamanho) {
    /* TODO:
       aloque a estrutura e o vetor de ponteiros.
       Inicialize todos os buckets com NULL.
    */
    return NULL;
}

void insere_hash(HashChaining *h, int chave) {
    /* TODO:
       1. calcule a posição;
       2. se o bucket já estiver ocupado, conte uma colisão;
       3. crie um novo nó;
       4. insira-o na lista.
    */
}

int busca_hash(
    const HashChaining *h,
    int chave,
    long long *comparacoes
) {
    /* TODO:
       percorra apenas a lista do bucket correspondente.
       Atualize *comparacoes a cada chave examinada.
       Retorne 1 se encontrar e 0 caso contrário.
    */
    return 0;
}

void imprime_hash(const HashChaining *h) {
    /* TODO:
       imprima cada bucket e sua lista.
    */
}

void libera_hash(HashChaining *h) {
    /* TODO:
       libere todos os nós, o vetor de buckets
       e a própria estrutura.
    */
}

int main(void) {
    const int tamanho = 10;
    const int n_chaves = 50;

    HashChaining *h = cria_hash(tamanho);
    if (!h)
        return 1;

    srand(42);

    int chaves[n_chaves];

    for (int i = 0; i < n_chaves; i++) {
        chaves[i] = 10 + rand() % 991;
        insere_hash(h, chaves[i]);
    }

    imprime_hash(h);

    printf("\nQuantidade armazenada: %d\n", h->quantidade);
    printf("Colisoes: %lld\n", h->colisoes);
    printf("Fator de carga: %.2f\n",
           (double)h->quantidade / h->tamanho);

    long long comparacoes = 0;
    int encontrou = busca_hash(h, chaves[10], &comparacoes);

    printf("\nBusca pela chave %d: %s | comparacoes=%lld\n",
           chaves[10],
           encontrou ? "encontrada" : "nao encontrada",
           comparacoes);

    libera_hash(h);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    hash_chaining_divisao.c \
    -o hash_chaining_divisao

!./hash_chaining_divisao

## Questões — chaining

1. Por que uma tabela com 10 buckets consegue armazenar 50 chaves?
2. O que significa fator de carga \(\alpha = 5\) em chaining?
3. Qual o melhor caso de busca?
4. O que aconteceria se todas as chaves fossem mapeadas para o mesmo bucket?
5. Nesse pior caso, qual estrutura a tabela hash praticamente se torna?

# Exercício Prático 7 — Função hash por folding

No método de **folding**, a chave é dividida em blocos e os blocos são combinados.

Neste notebook usaremos blocos de **dois dígitos decimais**.

Exemplo:

```text
chave = 123456

blocos:
12, 34, 56

soma = 12 + 34 + 56 = 102

posição = 102 % m
```

Uma maneira conveniente de implementar isso numericamente em C é extrair os blocos da direita para a esquerda:

```text
bloco = chave % 100
chave = chave / 100
```

Repita até que a chave seja zero.

## Sua tarefa

Implemente:

```c
int hash_folding(int chave, int m);
```

e compare a distribuição do folding com a distribuição obtida pelo método da divisão usando **as mesmas chaves**.

In [ ]:
%%writefile comparar_funcoes_hash.c
#include <stdio.h>
#include <stdlib.h>

#define M 10
#define N_CHAVES 50

int hash_divisao(int chave, int m) {
    return chave % m;
}

int hash_folding(int chave, int m) {
    /* TODO:
       divida a chave em blocos de dois dígitos,
       some os blocos e aplique % m.
    */
    return 0;
}

int main(void) {
    int divisao[M] = {0};
    int folding[M] = {0};

    srand(42);

    for (int i = 0; i < N_CHAVES; i++) {
        int chave = 10 + rand() % 991;

        int p1 = hash_divisao(chave, M);
        int p2 = hash_folding(chave, M);

        divisao[p1]++;
        folding[p2]++;
    }

    FILE *arq = fopen("comparacao_funcoes_hash.csv", "w");
    if (!arq)
        return 1;

    fprintf(arq, "posicao,divisao,folding\n");

    for (int i = 0; i < M; i++)
        fprintf(arq, "%d,%d,%d\n",
                i, divisao[i], folding[i]);

    fclose(arq);

    printf("Arquivo comparacao_funcoes_hash.csv criado.\n");
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    comparar_funcoes_hash.c \
    -o comparar_funcoes_hash

!./comparar_funcoes_hash

## Python — barras lado a lado

O `pandas` facilita a leitura, mas o gráfico continua sendo construído explicitamente.

Para colocar duas barras lado a lado, precisamos deslocar suas posições no eixo x.

`numpy.arange()` cria a sequência de posições:

```text
0, 1, 2, ..., 9
```

Depois usamos pequenas diferenças de posição para separar as barras.

Aqui o Python é útil porque a distribuição espacial é muito mais fácil de perceber visualmente do que em uma longa tabela de números.

In [ ]:
import numpy as np

dist_hash = pd.read_csv("comparacao_funcoes_hash.csv")

display(dist_hash)

x = np.arange(len(dist_hash))
largura = 0.38

plt.figure(figsize=(9, 5))

plt.bar(
    x - largura / 2,
    dist_hash["divisao"],
    width=largura,
    label="divisão"
)

plt.bar(
    x + largura / 2,
    dist_hash["folding"],
    width=largura,
    label="folding"
)

plt.xticks(x, dist_hash["posicao"])
plt.xlabel("Posição da tabela")
plt.ylabel("Quantidade de chaves")
plt.title("Distribuição: divisão × folding")
plt.legend()
plt.show()

## Como avaliar uma distribuição?

Não conclua que uma função é "melhor" apenas porque uma barra ficou menor.

Observe:

- concentração em poucos buckets;
- buckets vazios;
- maior bucket;
- diferença entre maior e menor ocupação;
- comportamento com outras sementes e outros conjuntos de chaves.

Uma boa função hash procura distribuir as chaves de maneira razoavelmente uniforme e deve ser barata de calcular.

# Exercício Prático 8 — Folding com chaining

Agora adapte sua implementação de chaining para usar:

```c
hash_folding(chave, tamanho)
```

em vez de `hash_divisao`.

Use exatamente a mesma semente:

```c
srand(42);
```

e as mesmas 50 chaves.

Compare:

- número de colisões;
- tamanho de cada cadeia;
- maior cadeia;
- custo médio de buscas bem-sucedidas;
- custo médio de buscas malsucedidas.

## Melhoria importante

Não crie a lista de colisões em Python. O chaining deve ser uma **lista ligada implementada em C**, pois ponteiros e alocação dinâmica fazem parte do objetivo desta atividade.

In [ ]:
%%writefile hash_chaining_folding.c
#include <stdio.h>
#include <stdlib.h>

typedef struct No {
    int chave;
    struct No *prox;
} No;

typedef struct {
    No **buckets;
    int tamanho;
    int quantidade;
    long long colisoes;
} HashChaining;

int hash_folding(int chave, int m) {
    /* TODO: reutilize sua implementação do exercício anterior. */
    return 0;
}

/* TODO:
   Reutilize/adapte as funções:
   - cria_hash
   - insere_hash
   - busca_hash
   - imprime_hash
   - libera_hash

   A diferença principal é a função de espalhamento utilizada.
*/

int main(void) {
    /*
     * TODO:
     * - tabela de tamanho 10;
     * - gere as mesmas 50 chaves com srand(42);
     * - insira todas as chaves;
     * - imprima a tabela;
     * - informe colisões e fator de carga;
     * - realize algumas buscas.
     */

    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    hash_chaining_folding.c \
    -o hash_chaining_folding

!./hash_chaining_folding

# Exercício Prático 9 — Folding + sondagem quadrática

Agora utilize **endereçamento aberto**.

Não haverá listas ligadas. Todas as chaves ficam diretamente no vetor da tabela.

Uma sequência comum de sondagem quadrática é:

\[
h_i(k) = (h(k) + i^2) \bmod m
\]

com:

```text
i = 0, 1, 2, ...
```

Assim, se a posição inicial estiver ocupada, testamos novas posições.

---

## Tabela de tamanho 100

Mantenha inicialmente o tamanho 100 do exercício original e utilize as mesmas 50 chaves.

### Atenção

A sondagem quadrática **não garante que todas as posições serão visitadas** para qualquer valor de `m`. Portanto, uma inserção pode falhar mesmo que ainda existam células vazias.

Isso é parte do que queremos observar.

### Estrutura sugerida

Use um vetor de `int` e um valor sentinela, por exemplo:

```c
#define VAZIO -1
```

Como nossas chaves são positivas, `-1` pode indicar uma célula vazia.

Conte:

- colisões;
- número total de sondagens;
- inserções que falharam.

In [ ]:
%%writefile hash_quadratica.c
#include <stdio.h>
#include <stdlib.h>

#define VAZIO -1

typedef struct {
    int *dados;
    int tamanho;
    int quantidade;
    long long colisoes;
    long long sondagens;
    int falhas;
} HashQuadratica;

int hash_folding(int chave, int m) {
    /* TODO: implemente ou reutilize o folding. */
    return 0;
}

HashQuadratica *cria_hash_quadratica(int tamanho) {
    /* TODO:
       aloque a estrutura e o vetor.
       Inicialize todas as posições com VAZIO.
    */
    return NULL;
}

int insere_quadratica(HashQuadratica *h, int chave) {
    /* TODO:
       Para i = 0 ... tamanho-1:
          pos = (hash_folding(chave,tamanho) + i*i) % tamanho

       Conte as sondagens.
       Se encontrar uma posição vazia, insira e retorne 1.
       Se a primeira tentativa ou uma tentativa posterior estiver
       ocupada, contabilize adequadamente a colisão.

       Se nenhuma posição válida for encontrada, incremente falhas
       e retorne 0.
    */
    return 0;
}

int busca_quadratica(
    const HashQuadratica *h,
    int chave,
    long long *sondagens
) {
    /* TODO:
       repita a mesma sequência de sondagem usada na inserção.
       Interrompa se encontrar VAZIO.
       Retorne 1/0.
    */
    return 0;
}

void imprime_hash_quadratica(const HashQuadratica *h) {
    /* TODO */
}

void libera_hash_quadratica(HashQuadratica *h) {
    /* TODO */
}

int main(void) {
    const int tamanho = 100;
    const int n_chaves = 50;

    HashQuadratica *h =
        cria_hash_quadratica(tamanho);

    if (!h)
        return 1;

    srand(42);

    int chaves[n_chaves];

    for (int i = 0; i < n_chaves; i++) {
        chaves[i] = 10 + rand() % 991;
        insere_quadratica(h, chaves[i]);
    }

    imprime_hash_quadratica(h);

    printf("\nInseridos: %d\n", h->quantidade);
    printf("Colisoes: %lld\n", h->colisoes);
    printf("Sondagens: %lld\n", h->sondagens);
    printf("Falhas: %d\n", h->falhas);
    printf("Fator de carga: %.3f\n",
           (double)h->quantidade / h->tamanho);

    long long s = 0;
    int encontrou =
        busca_quadratica(h, chaves[10], &s);

    printf("Busca por %d: %s | sondagens=%lld\n",
           chaves[10],
           encontrou ? "encontrada" : "nao encontrada",
           s);

    libera_hash_quadratica(h);
    return 0;
}

In [ ]:
!gcc -Wall -Wextra -O2 -std=c11 \
    hash_quadratica.c \
    -o hash_quadratica

!./hash_quadratica

## Desafio — tamanho 100 × tamanho 101

Repita a sondagem quadrática com:

```text
m = 100
m = 101
```

mantendo as mesmas chaves.

O número 101 é primo. Compare:

- falhas de inserção;
- colisões;
- sondagens;
- ocupação.

Não generalize a partir de uma única execução, mas use o experimento para investigar como o tamanho da tabela interage com a sequência de sondagem.

# Exercício Prático 10 — Experimento final: efeito do fator de carga

Esta atividade integra todos os conceitos de hash.

Use uma tabela de tamanho:

```text
m = 101
```

e teste aproximadamente os fatores de carga:

```text
0.25
0.50
0.70
0.85
```

Para cada fator de carga:

1. gere chaves aleatórias;
2. construa a estrutura;
3. realize buscas bem-sucedidas;
4. realize buscas malsucedidas;
5. registre o custo médio.

Compare pelo menos:

### Estratégia A

**divisão + chaining**

### Estratégia B

**folding + chaining**

### Estratégia C

**folding + sondagem quadrática**

---

## CSV sugerido

Seu programa C deverá gerar:

```text
resultados_hash.csv
```

com:

```text
estrategia,alpha,tipo_busca,repeticao,custo_medio,colisoes
```

Para chaining, `custo_medio` pode representar o número de nós examinados.

Para sondagem quadrática, `custo_medio` pode representar o número de posições sondadas.

### Importante

O significado de "colisão" deve permanecer consistente dentro de cada estratégia e ser explicado no relatório.

In [ ]:
%%writefile benchmark_hash.c
#include <stdio.h>

/*
 * EXERCÍCIO INTEGRADOR
 *
 * Reutilize as estruturas e funções implementadas nos exercícios
 * anteriores.
 *
 * Sugestão:
 *
 * 1. Crie funções separadas para executar um experimento com:
 *      - divisão + chaining
 *      - folding + chaining
 *      - folding + sondagem quadrática
 *
 * 2. Use:
 *
 *      m = 101
 *
 * 3. Para cada alpha:
 *
 *      0.25, 0.50, 0.70, 0.85
 *
 *    calcule:
 *
 *      n = alpha * m
 *
 * 4. Faça várias repetições usando sementes diferentes.
 *
 * 5. Realize a mesma quantidade de buscas com sucesso
 *    e sem sucesso.
 *
 * 6. Grave resultados_hash.csv no formato:
 *
 * estrategia,alpha,tipo_busca,repeticao,custo_medio,colisoes
 *
 * Não faça a análise dos dados em C.
 * O programa C deve produzir os dados experimentais;
 * o Python será utilizado depois para a análise.
 */

int main(void) {
    /* TODO: implemente o benchmark final. */

    printf("Implemente o benchmark e gere resultados_hash.csv\n");
    return 0;
}

# Python — análise do benchmark de hash

Execute esta parte **somente depois** que seu programa C tiver criado `resultados_hash.csv`.

Primeiro verificamos se o arquivo existe com `Path`.

`Path` faz parte da biblioteca `pathlib` e facilita o trabalho com caminhos e arquivos.

In [ ]:
from pathlib import Path

arquivo_hash = Path("resultados_hash.csv")

print("Arquivo encontrado?", arquivo_hash.exists())

## Leitura e resumo

Se o arquivo existir:

```python
pd.read_csv(...)
```

transforma os resultados em um `DataFrame`.

Depois usamos `groupby` para calcular o custo médio das repetições.

In [ ]:
if arquivo_hash.exists():
    hash_df = pd.read_csv(arquivo_hash)

    display(hash_df.head())

    resumo_hash = (
        hash_df.groupby(
            ["estrategia", "alpha", "tipo_busca"]
        )
        .agg(
            custo_medio=("custo_medio", "mean"),
            colisoes_medias=("colisoes", "mean"),
        )
        .reset_index()
    )

    display(resumo_hash)
else:
    print(
        "Execute primeiro o benchmark em C "
        "para gerar resultados_hash.csv."
    )

## Gráfico: custo de busca × fator de carga

O fator de carga é uma variável ordenada. Por isso, um gráfico de linhas ajuda a visualizar a tendência conforme \(\alpha\) aumenta.

O código abaixo gera um gráfico para buscas bem-sucedidas e outro para buscas malsucedidas.

In [ ]:
if arquivo_hash.exists():
    for tipo in resumo_hash["tipo_busca"].unique():
        dados = resumo_hash[
            resumo_hash["tipo_busca"] == tipo
        ]

        plt.figure(figsize=(8, 5))

        for estrategia in dados["estrategia"].unique():
            parte = dados[
                dados["estrategia"] == estrategia
            ]

            plt.plot(
                parte["alpha"],
                parte["custo_medio"],
                marker="o",
                label=estrategia
            )

        plt.xlabel("Fator de carga (alpha)")
        plt.ylabel("Custo médio da busca")
        plt.title(f"Hash — busca {tipo}")
        plt.legend()
        plt.grid()
        plt.show()

# Questões finais para discussão

## Busca

1. Em quais situações busca sequencial ainda pode ser uma escolha razoável?
2. Por que busca binária é \(O(\log n)\)?
3. Em um sistema que recebe apenas **uma consulta**, ordenar antes da busca costuma compensar?
4. E em um sistema que realizará milhares de consultas sobre a mesma coleção?
5. Por que número de inspeções pode ser uma métrica mais didática que tempo?

## Hash

6. O que é uma colisão?
7. Por que colisões não significam necessariamente que a função hash está errada?
8. Qual é a diferença estrutural entre chaining e endereçamento aberto?
9. Como o fator de carga afeta cada estratégia?
10. Por que uma busca malsucedida pode ser mais cara do que uma busca bem-sucedida?
11. O que ocorre com chaining se todas as chaves forem enviadas para o mesmo bucket?
12. Por que a escolha do tamanho da tabela pode ser relevante em sondagem quadrática?
13. Nos seus experimentos, divisão ou folding apresentou distribuição mais uniforme? Essa conclusão seria válida para qualquer conjunto de chaves?

## Pensamento crítico

14. Uma tabela hash é sempre melhor que busca binária?
15. Quando você escolheria:
    - vetor + busca sequencial;
    - vetor ordenado + busca binária;
    - tabela hash?
16. Quais operações, além de busca, deveriam ser consideradas antes de escolher uma estrutura de dados?

# Desafio opcional — pequeno sistema de cadastro

Implemente em C um cadastro simples de registros:

```c
typedef struct {
    int id;
    char nome[80];
} Registro;
```

A chave será `id`.

Crie duas versões:

### Versão 1
Vetor de registros + busca sequencial.

### Versão 2
Tabela hash com chaining.

Implemente:

- inserir;
- buscar por `id`;
- remover;
- imprimir.

Depois gere uma carga sintética de registros e compare o número médio de elementos examinados nas buscas.

Esse desafio aproxima os algoritmos do tipo de problema encontrado em sistemas de informação reais.